In [ ]:
///- --package .

#### cooldowns (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:| spell_group =
    | AttackGroup
    | HealingGroup
    | SupportGroup

:! attack = ()

:! healing = ()

:! support = ()

:| spell_id g =
    | StrikeSpell :: spell_id attack
    | LightHealingSpell :: spell_id healing
    | WoundCleansingSpell :: spell_id healing
    | HasteSpell :: spell_id support

:= group_of \* g. (id : spell_id g) : spell_group =
    ?= id ?|
    | StrikeSpell => AttackGroup
    | LightHealingSpell => HealingGroup
    | WoundCleansingSpell => HealingGroup
    | HasteSpell => SupportGroup

:== spell g = { id : spell_id g; own_ms : i32; group_ms : i32 }

:== timers =
    {
        attack_ready_at : i32
        healing_ready_at : i32
        support_ready_at : i32
        strike_ready_at : i32
        light_healing_ready_at : i32
        wound_cleansing_ready_at : i32
        haste_ready_at : i32
    }

:= idle_timers () : timers =
    {
        attack_ready_at = 0
        healing_ready_at = 0
        support_ready_at = 0
        strike_ready_at = 0
        light_healing_ready_at = 0
        wound_cleansing_ready_at = 0
        haste_ready_at = 0
    }

:= group_ready_at (group : spell_group) (held : timers) : i32 =
    ?= group ?|
    | AttackGroup => held::attack_ready_at
    | HealingGroup => held::healing_ready_at
    | SupportGroup => held::support_ready_at

:= own_ready_at \* g. (id : spell_id g) (t : timers) : i32 =
    ?= id ?|
    | StrikeSpell => t::strike_ready_at
    | LightHealingSpell => t::light_healing_ready_at
    | WoundCleansingSpell => t::wound_cleansing_ready_at
    | HasteSpell => t::haste_ready_at

:= with_strike (at : i32) (t : timers) : timers = { t ?| strike_ready_at = at }

:= with_light_healing (at : i32) (t : timers) : timers = { t ?| light_healing_ready_at = at }

:= with_wound_cleansing (at : i32) (t : timers) : timers = { t ?| wound_cleansing_ready_at = at }

:= with_haste (at : i32) (t : timers) : timers = { t ?| haste_ready_at = at }

:= with_own_ready_at \* g. (id : spell_id g) (at : i32) (t : timers) : timers =
    ?= id ?|
    | StrikeSpell => with_strike at t
    | LightHealingSpell => with_light_healing at t
    | WoundCleansingSpell => with_wound_cleansing at t
    | HasteSpell => with_haste at t

:= with_group_ready_at (group : spell_group) (at : i32) (t : timers) : timers =
    ?= group ?|
    | AttackGroup => { t ?| attack_ready_at = at }
    | HealingGroup => { t ?| healing_ready_at = at }
    | SupportGroup => { t ?| support_ready_at = at }

:= later (a : i32) (b : i32) : i32 =
    ? a < b ?> b |: a

:= ready_at \* g. (cast : spell g) (held : timers) : i32 =
    later (group_ready_at (group_of cast::id) held) (own_ready_at cast::id held)

:= can_cast \* g. (now : i32) (cast : spell g) (held : timers) : bool =
    ready_at cast held <= now

:= after_cast \* g. (now : i32) (cast : spell g) (held : timers) : timers =
    := own_until = later (own_ready_at cast::id held) (now + cast::own_ms)
    with_own_ready_at cast::id own_until held |> with_group_ready_at (group_of cast::id) (now + cast::group_ms)

:= strike_spell () : spell attack = { id = StrikeSpell; own_ms = 2000; group_ms = 2000 }

:= light_healing () : spell healing = { id = LightHealingSpell; own_ms = 1000; group_ms = 1000 }

:= haste () : spell support = { id = HasteSpell; own_ms = 2000; group_ms = 2000 }

:= after_strike_at (now : i32) : timers = idle_timers () |> after_cast now (strike_spell ())

#### fresh_timers_allow_any_spell (test)

In [1]:
///- --test --symbol
///> c

:= ~now = 0i32
can_cast now (strike_spell ()) (idle_timers ()) |> _assert_eq !+

__assert_eq { expected = true }

#### attack_blocks_its_group_until_it_expires (test)

In [2]:
///- --test --symbol
///> c

:= ~now = 1000i32
:= h = idle_timers () |> after_cast now (strike_spell ())
can_cast (now + 1999) (strike_spell ()) h |> _assert_eq !-
can_cast (now + 2000) (strike_spell ()) h |> _assert_eq !+

__assert_eq { expected = false }
__assert_eq { expected = true }

#### attack_does_not_block_healing_group (test)

In [3]:
///- --test --symbol
///> c

:= ~now = 1000i32
:= held = idle_timers () |> after_cast now (strike_spell ())
group_ready_at HealingGroup held |> _assert_eq 0i32

__assert_eq { expected = 0 }

#### own_cooldown_gates_only_the_same_spell_like_canary (test)

In [4]:
///- --test --symbol
///> c

:= ~now = 1000i32
:= held = after_strike_at now
can_cast (now + 500) (light_healing ()) held |> _assert_eq !+
can_cast (now + 500) (strike_spell ()) held |> _assert_eq !-

__assert_eq { expected = true }
__assert_eq { expected = false }

#### the_own_cooldown_ends_with_the_group_cooldown (test)

In [5]:
///- --test --symbol
///> c

:= ~now = 1000i32
can_cast (now + 2000) (strike_spell ()) (after_strike_at now) |> _assert_eq !+

__assert_eq { expected = true }

#### support_sets_only_support_group (test)

In [6]:
///- --test --symbol
///> c

:= ~now = 0i32
:= held = idle_timers () |> after_cast now (haste ())
held::support_ready_at |> _assert_eq 2000i32
held::attack_ready_at |> _assert_eq 0i32

__assert_eq { expected = 2000 }
__assert_eq { expected = 0 }

#### attack_cooldown_on_gpu (test)

In [7]:
///- --test --symbol
///> wgsl

:= ~now = 1000i32
:= held, strike = after_strike_at now, strike_spell ()
? can_cast (now + 1999) strike held = !- && can_cast (now + 2000) strike held && held::healing_ready_at = 0i32
?> 0i32
|: 1i32

#### healing_right_after_a_strike_is_allowed_like_canary (test)

In [8]:
///- --test --symbol
///> c

:= ~now = 1000i32
:= held = after_cast now (strike_spell ()) (idle_timers ())
can_cast now (light_healing ()) held |> _assert_eq !+
can_cast now (strike_spell ()) held |> _assert_eq !-

__assert_eq { expected = true }
__assert_eq { expected = false }

#### a_strike_cannot_be_typed_as_a_healing_spell (test)

In [9]:
///- --test --symbol --fails "Unification failure"
///> c

:= s : spell healing = { id = StrikeSpell; own_ms = 2000; group_ms = 2000 }
s::own_ms

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/tibia/tibia/src/cooldowns.spi:162:35 (main.spi:107:35): ETY-TermError: Unification failure. Got:      spell_id healing Expected: spell_id attack In the definition: spiral_cells_test.
